# Bird-detection dataset build

Self-contained build of the two-class bird / no_bird collection for `22_detection_train`. Discovers the area species and downloads them (the bird pool), pulls AudioSet ambient noise (the no_bird pool, with the BirdNET no-bird harvest), then assembles `datasets/detection_<lat>_<lon>/`. Downloads cached and shared with the scale build.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import os

import pyrootutils

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator="pyproject.toml",
    pythonpath=True,
    dotenv=True,
)

## Parameters

In [ ]:
from building.polychirp_training import detection_slug

# Discovered area species form the bird pool; slug is detection_<lat>_<lon>.
LAT, LON = 48.8566, 2.3522   # Paris
RADIUS_KM = 50

# Area discovery (matches the scale build).
MIN_AREA_RECORDINGS = 10
ANALYSIS_MAX_BIRDNET = 2000
ANALYSIS_MIN_CONF = 0.85

# Per-species download budget (per source); bird/ keeps PER_PROVIDER per provider.
CLIPS_PER_SOURCE = 300

# AudioSet ambient noise goes to no_bird (with the BirdNET no-bird harvest).
AS_CLIPS_PER_CLASS = 250
AS_MAX_TOTAL_CLIPS = 15000

# Assembly caps.
PER_SPECIES = 500
PER_PROVIDER = 250

DETECTION_COLLECTION = detection_slug(LAT, LON)
print(f"detection collection: {DETECTION_COLLECTION}")


## Discover area species

In [ ]:
from building.analysis import (
    area_audio_cache_dir,
    bbox_from_radius,
    birdnet_area_analysis,
    combined_species_table,
    fetch_area_recordings,
    recordings_to_df,
)

bbox = bbox_from_radius(LAT, LON, RADIUS_KM)
area_df = recordings_to_df(await fetch_area_recordings(bbox))
bn = await birdnet_area_analysis(
    area_df,
    max_recordings=ANALYSIS_MAX_BIRDNET,
    min_confidence=ANALYSIS_MIN_CONF,
    cache_dir=area_audio_cache_dir(LAT, LON, RADIUS_KM),
)
combined = combined_species_table(
    area_df, bn["detections"], min_recordings=MIN_AREA_RECORDINGS
)
all_species = combined["scientific_name"].tolist()
print(f"{len(all_species)} area species -> bird pool")

## Download area species (bird pool)

In [ ]:
from building.data.download import download_and_process
from building.data.sources import XCSource, EBirdSource

with XCSource() as xc, EBirdSource() as eb:
    for sp in all_species:
        await download_and_process(sp, CLIPS_PER_SOURCE, [xc, eb])

## Download AudioSet ambient (no_bird pool)

In [ ]:
from building.data.audioset import AudioSetConfig, stream_download_audioset_async

as_cfg = AudioSetConfig(
    clips_per_class=AS_CLIPS_PER_CLASS,
    max_total_clips=AS_MAX_TOTAL_CLIPS,
)
results = await stream_download_audioset_async(as_cfg)

## Assemble bird / no_bird

In [ ]:
from building.data.dataset import build_detection_dataset

dataset_path = build_detection_dataset(
    DETECTION_COLLECTION, per_species=PER_SPECIES, per_provider=PER_PROVIDER
)
print(dataset_path)

## Sample counts per class and split

In [ ]:
import pandas as pd

from building.data.dataset import DATASETS_DIR, SPLIT_NAMES

root = DATASETS_DIR / DETECTION_COLLECTION
cols = ["train", "val", "test"]
classes = sorted(d.name for d in (root / SPLIT_NAMES[0]).iterdir() if d.is_dir())
counts = pd.DataFrame(
    [[len(list((root / s / c).glob("*.wav"))) for s in SPLIT_NAMES] for c in classes],
    index=classes, columns=cols,
)
counts["total"] = counts.sum(axis=1)
counts.loc["TOTAL"] = counts.sum(axis=0)

(
    counts.style
    .background_gradient(cmap="Blues", subset=(counts.index[:-1], cols))
    .format("{:,}")
    .set_caption(f"Sample counts per class and split  ({DETECTION_COLLECTION})")
    .set_table_styles([
        {"selector": "caption", "props": [("font-weight", "bold"), ("font-size", "110%"), ("padding", "6px 0")]},
        {"selector": "th", "props": [("text-align", "center")]},
        {"selector": "td", "props": [("text-align", "right"), ("padding", "4px 10px")]},
    ])
)
